### Weniger Bias, mehr Rendite? Hybrides Modell aus Large Language Models und Machine Learning zur Analyse von Earnings Calls
**Bachelorarbeit an der Universität St.Gallen (HSG)**\
[© 2026 Ruben Cardell](https://polyformproject.org/licenses/noncommercial/1.0.0)

---

# Preprocessing – Kontrollversuch

Dieses Notebook bereitet die Daten für den Kontrollversuch zum Distraction Bias
auf. Es bildet den Datensatz aus Versuch 2 nach, verzichtet jedoch bewusst auf
die Anonymisierung der Unternehmensnamen. Der Vergleich der LLM-Ausgaben beider
Läufe misst, welcher Anteil der Ausgabe auf unternehmensspezifisches Vorwissen
zurückgeht.

Schritte:

1. Stichprobengrösse über eine Power-Analyse bestimmen
2. Frage-Antwort-Teile ohne Anonymisierung extrahieren und Stichprobe ziehen
3. Token-Anzahl der längsten Eingabe prüfen

In [1]:
# Google Drive einbinden (Ausführung in Google Colab)
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

Mounted at /content/drive


## 1 Stichprobengrösse ermitteln

Der Kontrollversuch erfordert einen weiteren ressourcenintensiven LLM-Lauf und
wird daher auf einer Stichprobe durchgeführt. Deren Grösse wird vorab über eine
Power-Analyse für den gepaarten t-Test bestimmt.

In [ ]:
# Power-Analyse für den gepaarten t-Test
from statsmodels.stats.power import TTestPower

analysis = TTestPower()

In [10]:
# Bonferroni-Korrektur über 11 Tests: den aggregierten Test, die neun
# Einzeldimensionen und die Direktprognose
alpha = 0.05 / 11

# Benötigte Stichprobengrösse je Effektstärke bei einer Teststärke von 0.80
for d in [0.1, 0.15, 0.2, 0.3, 0.4]:
    n = analysis.solve_power(effect_size=d, alpha=alpha, power=0.80, alternative="two-sided")
    print(f"d={d}: n={n:.0f}")

d=0.1: n=1358
d=0.15: n=606
d=0.2: n=342
d=0.3: n=154
d=0.4: n=89


## 2 Datensatz vorbereiten

Der Kontrollversuch verwendet denselben Datensatz wie Versuch 2, jedoch **ohne**
Anonymisierung. Die beiden Läufe unterscheiden sich damit ausschliesslich in der
Anonymisierung der Unternehmensnamen.

In [24]:
# Rohdatensatz: Earnings-Call-Transkripte des S&P 500 (2005–2025)
df = pd.read_parquet('/content/drive/MyDrive/Bachelorarbeit/data/raw_hf.parquet')

In [25]:
# Erste Zeilen zur Sichtprüfung
df.head()

,symbol,company_name,company_id,year,quarter,date,structured_content,earnings_date,episode_id
0,A,"Agilent Technologies, Inc.",154924.0,2006,1,2006-02-13 14:45:00,[{'speaker': 'Agilent Technologies Incorporate...,2006-02-13,A_2006_Q1
1,A,"Agilent Technologies, Inc.",154924.0,2006,2,2006-05-15 16:30:00,"[{'speaker': 'Executives', 'text': 'Hilliard C...",2006-05-15,A_2006_Q2
2,A,"Agilent Technologies, Inc.",154924.0,2006,3,2006-08-14 16:30:00,"[{'speaker': 'Executives', 'text': 'Hilliard C...",2006-08-14,A_2006_Q3
3,A,"Agilent Technologies, Inc.",154924.0,2006,4,2006-11-14 08:00:00,"[{'speaker': 'Executives', 'text': 'Hilliard C...",2006-11-14,A_2006_Q4
4,A,"Agilent Technologies, Inc.",154924.0,2007,1,2007-02-15 16:30:00,"[{'speaker': 'TRANSCRIPT SPONSOR', 'text': ''}...",2007-02-15,A_2007_Q1


In [26]:
# Spalten, Datentypen und fehlende Werte prüfen
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33362 entries, 0 to 33361
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   symbol              33362 non-null  object 
 1   company_name        24377 non-null  object 
 2   company_id          24377 non-null  float64
 3   year                33362 non-null  int64  
 4   quarter             33362 non-null  int64  
 5   date                33362 non-null  object 
 6   structured_content  33362 non-null  object 
 7   earnings_date       33362 non-null  object 
 8   episode_id          33362 non-null  object 
dtypes: float64(1), int64(2), object(6)
memory usage: 2.3+ MB


In [17]:
def extract_qa_pairs(structured_content):
    """Extrahiert die Frage-Antwort-Paare aus einem Earnings-Call-Transkript.

    Der Operator kündigt in Earnings Calls jeden Analysten an und ist in den
    Transkripten einheitlich benannt. Seine Turns dienen daher als Trennmarken
    zwischen den Frage-Antwort-Paaren. Die vorbereitete Einleitung des
    Managements wird verworfen, da sie andere sprachliche Signale enthält als
    die spontanen Antworten im Frage-Antwort-Teil.

    Rückgabe: Liste von Segmenten, je Segment eine Liste von Sprecher-Turns.
    Leere Liste, falls kein Operator ausgewiesen ist (741 Transkripte).
    """
    try:
        turns = structured_content.tolist()
        assert all("speaker" in t and "text" in t for t in turns)
    except Exception:
        return []

    # Metadaten-Turns entfernen, die nur die Teilnehmerliste aufzählen
    IGNORED_SPEAKERS = {"executives", "analysts", "presentation"}
    turns = [t for t in turns if str(t["speaker"]).strip().lower() not in IGNORED_SPEAKERS]

    # Segmente an den Operator-Turns trennen; der Operator selbst wird verworfen
    segments, current = [], []
    for t in turns:
        if str(t["speaker"]).strip().lower() == "operator":
            segments.append(current)
            current = []

            # Ausnahme: Enthält der Operator-Turn eine Frage, gehört er zum Segment
            if "?" in str(t["text"]):
              current.append(t)
        else:
            current.append(t)
    segments.append(current)

    # Das erste nicht-leere Segment ist immer die Einleitung des Managements und
    # wird übersprungen, auch wenn es zufällig eine (rhetorische) Frage enthält.
    first_nonempty = next((i for i, seg in enumerate(segments) if seg), None)
    if first_nonempty is None:
        return []

    # Nur Segmente mit mindestens einer Frage und einer Antwort behalten
    return [seg for seg in segments[first_nonempty + 1:] if len(seg) > 1]

In [27]:
from tqdm.notebook import tqdm

In [28]:
stats_rows = []
new_rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    # Kein Anonymisierungsschritt: die Transkripte bleiben im Original
    qa_pairs = extract_qa_pairs(row["structured_content"])

    # Jedes Segment als Fliesstext "Sprecher: Text" für den User-Prompt formatieren
    prompts = ['\n'.join(f'{turn["speaker"]}: {turn["text"]}' for turn in seg) for seg in qa_pairs]

    # Eine Zeile je Frage-Antwort-Paar; sequence nummeriert die Paare im Call
    if len(qa_pairs) != 0:
        for seq, qa_pair in enumerate(prompts, start=1):
            new_rows.append({
                "episode_id": row["episode_id"],
                "sequence": seq,
                "qa_pair": qa_pair,
            })

new_df = pd.DataFrame(new_rows)

  0%|          | 0/33362 [00:00<?, ?it/s]

In [29]:
# Stichprobe der extrahierten Frage-Antwort-Paare
new_df[112360:112370]

,episode_id,sequence,qa_pair
112360,EL_2012_Q2,7,Constance Marie Maneaty - BMO Capital Markets ...
112361,EL_2012_Q2,8,"Linda Bolton-Weiser - Caris & Company, Inc., R..."
112362,EL_2012_Q2,9,"Javier Escalante - Consumer Edge Research, LLC..."
112363,EL_2012_Q2,10,"John A. Faucher - JP Morgan Chase & Co, Resear..."
112364,EL_2012_Q2,11,"William Schmitz - Deutsche Bank AG, Research D..."
112365,EL_2012_Q3,1,"Christopher Ferrara - BofA Merrill Lynch, Rese..."
112366,EL_2012_Q3,2,"Neely J.N. Tamminga - Piper Jaffray Companies,..."
112367,EL_2012_Q3,3,"Mark S. Astrachan - Stifel, Nicolaus & Co., In..."
112368,EL_2012_Q3,4,David Wu - Telsey Advisory Group LLC: On Europ...
112369,EL_2012_Q3,5,Alice Beebe Longley - The Buckingham Research ...


In [30]:
# Alle Paare eines Transkripts zu einem Text zusammenfügen (wie in Versuch 2)
control_df = new_df.groupby('episode_id')['qa_pair'].apply('\n\n'.join).reset_index()

In [31]:
# Anzahl Transkripte und fehlende Werte prüfen
control_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32621 entries, 0 to 32620
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   episode_id  32621 non-null  object
 1   qa_pair     32621 non-null  object
dtypes: object(2)
memory usage: 509.8+ KB


In [32]:
# Vollständigen Frage-Antwort-Teil eines Transkripts prüfen
print(control_df.iloc[0]["qa_pair"])

Frank Boroch - BearStearns: Tom, if you could maybe give us a sense of how unit revenueperformed across the hubs in the quarter, that would be helpful.
Tom Horton: Our domestic unit revenue in the quarter was up 4.8%. As youbreak that down, it was up about 5.5% to TransCON, Dallaswas up about 2.8%, Chicago was up3.8%, Miami was up a little over 3%, and St. Louis was up almost 8%.
Frank Boroch - BearStearns: You alluded to the policy change, the expiration policychange coming in December. Can you give us a sense of the magnitude, therevenue impact you're expecting from the 18-month expiration period?
Tom Horton: We're still working on it. It will depend on the results ofthat work, but it will be measured in the tens of millions of dollars inrevenue.
Frank Boroch - BearStearns: Is there anything happening in the Atlantic division, interms of, your revenue seems to be underperforming a little bit the broaderindustry?
Tom Horton: I think that's a fair question. Our Trans-Atlanticperformanc

In [33]:
# Zufallsstichprobe in der oben bestimmten Grösse; fester Zufallsstartwert
# für die Reproduzierbarkeit
control_df = control_df.sample(n=1358, random_state=42)

In [34]:
# Stichprobengrösse kontrollieren
control_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1358 entries, 6528 to 19221
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   episode_id  1358 non-null   object
 1   qa_pair     1358 non-null   object
dtypes: object(2)
memory usage: 31.8+ KB


In [35]:
# Datensatz des Kontrollversuchs speichern (Eingabe für main_v4.py)
control_df.to_csv('/content/drive/MyDrive/Bachelorarbeit/data/prep_data_control.csv', index=False)

## 3 Token-Anzahl prüfen

In [48]:
from transformers import AutoTokenizer

MODEL_NAME = "google/gemma-4-E4B-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

In [49]:
# Längste Eingabe bestimmen; daraus wird MAX_MODEL_LEN in main_v4.py abgeleitet
longest_qa = control_df['qa_pair'].str.len().idxmax()

max_tokens = len(tokenizer.encode(control_df['qa_pair'].loc[longest_qa])) + 737  # zzgl. System-Prompt

print(max_tokens)

21012
